# 06 — End-to-end evaluation and API contract

## What is evaluated

This notebook tests the complete chain: profile mapping, planning, retrieval, evidence validation, answer status, citations and HTTP serialization. Positive cases are paired with deliberately unanswerable scope, entity and period cases.

## Reading guide, cell by cell

1. **Bootstrap** reloads the application and global artifact store, preventing stale valid QRT IDs from appearing unknown.
2. **Aggregate metrics** runs the golden set and asserts zero false completeness and perfect citation precision.
3. **Case table** exposes expected/actual status and covered fields; use it to diagnose misleading aggregates.
4. **Latency loop** measures ten warm in-process calls. p50 is the median and p95 the slow tail. Browser, network and deployment latency are excluded.
5. **HTTP contract** verifies health and query endpoints, claims and the six-document deployed library.

## Stored-run conclusion

All quality metrics are perfect on 11 curated cases and false completeness is 0. This passes the demo gate and is internally coherent, including negative entity/period/scope tests. It is not production evidence: the set is small and close to implemented profiles. Median latency is about 4.6 seconds—acceptable for the demonstration, but too slow at volume without batching, persistent vector search and stronger caching.

In [1]:
from pathlib import Path
import statistics
import sys
import time

root_hint = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_hint / 'notebooks'))
from helpers import bootstrap, display_table

ROOT = bootstrap()

# Reload the local application after corpus or code changes. Without this, a
# long-lived Jupyter kernel can retain an obsolete global ArtifactStore.
for module_name in list(sys.modules):
    if module_name == 'app' or module_name.startswith('app.'):
        del sys.modules[module_name]

sys.path.insert(0, str(ROOT / 'backend' / 'evals'))
from run_evals import evaluate
from app.domain.models import QuestionRequest, ScopeSelection
from app.services.engine import engine

In [2]:
report = await evaluate()
metrics = report['metrics']
assert metrics['false_completeness_rate'] == 0.0
assert metrics['citation_precision'] == 1.0
display_table([{'metric': name, 'value': value} for name, value in metrics.items()])

,metric,value
0,cases,14.0
1,profile_accuracy,1.0
2,status_accuracy,1.0
3,required_field_recall,1.0
4,citation_precision,1.0
5,false_completeness_rate,0.0


In [3]:
display_table([
    {
        'case': result['id'],
        'profil_ok': result['profile_ok'],
        'statut_attendu': result['expected_status'],
        'statut_obtenu': result['actual_status'],
        'champs_couverts': ', '.join(result['covered_fields']),
    }
    for result in report['results']
])

,case,profil_ok,statut_attendu,statut_obtenu,champs_couverts
0,qrt-prudential-coverage-complete,True,COMPLETE,COMPLETE,"eligible_own_funds_scr, group_scr, scr_coverag..."
1,qrt-prudential-coverage-complete-french,True,COMPLETE,COMPLETE,"eligible_own_funds_scr, group_scr, scr_coverag..."
2,qrt-wrong-period-not-found,True,NOT_FOUND,NOT_FOUND,
3,position-complete,True,COMPLETE,COMPLETE,"business_areas, group_equity, non_life_market_..."
4,customers-complete,True,COMPLETE,COMPLETE,"claims_satisfaction, insured_households, myfoy..."
5,health-complete,True,COMPLETE,COMPLETE,"active_countries, earned_premiums, employee_count"
6,wrong-scope-not-found,True,NOT_FOUND,NOT_FOUND,
7,turnover-unanswerable,True,NOT_FOUND,NOT_FOUND,
8,claim-count-unanswerable,True,NOT_FOUND,NOT_FOUND,
9,foyer-assurances-prudential-complete,True,COMPLETE,COMPLETE,"eligible_own_funds_scr, entity_scr, scr_covera..."


In [4]:
request = QuestionRequest(
    question='Quels indicateurs clients et opérationnels sont publiés pour 2025 ?',
    mode='deep',
    scope=ScopeSelection(document_ids=['foyer_annual_report_2025']),
)
measurements = []
for _ in range(10):
    started = time.perf_counter()
    result = await engine.answer(request)
    measurements.append((time.perf_counter() - started) * 1000)
ordered = sorted(measurements)
{
    'iterations': len(measurements),
    'p50_ms': round(statistics.median(measurements), 2),
    'p95_ms': round(ordered[min(len(ordered) - 1, int(len(ordered) * 0.95))], 2),
    'status': result.status,
}

{'iterations': 10, 'p50_ms': 4601.26, 'p95_ms': 4781.56, 'status': 'COMPLETE'}

In [5]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
health = client.get('/api/health')
answer = client.post('/api/query', json={
    'question': 'Quels chiffres clés sont publiés pour Global Health ?',
    'mode': 'deep',
    'scope': {'document_ids': ['foyer_annual_report_2025']},
})
assert health.status_code == 200
assert answer.status_code == 200
assert answer.json()['status'] == 'COMPLETE'
{
    'health': health.json(),
    'query_status': answer.json()['status'],
    'claims': len(answer.json()['claims']),
}

C:\Users\choun\Downloads\Prudential_Evidence_Lab_MVP_Source\prudential_evidence_lab\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


{'health': {'status': 'ok',
  'corpus_version': 'foyer-public-demo-2026-08-21.1',
  'documents': 6,
  'chunks': 1918},
 'query_status': 'COMPLETE',
 'claims': 3}

### Interprétation responsable

Un jeu initial de six questions valide le câblage du démonstrateur ; il ne prouve pas une qualité de production. Étendre ensuite les évaluations par type de document, entité, période, tableau, ambiguïté et question non répondable.